# NYC Airbnb — EDA, Cleaning & Power BI Prep

End-to-end notebook: data cleaning, feature engineering, EDA, and loading to PostgreSQL for Power BI.

Every cleaning decision is explained inline — the goal is to show *reasoning*, not just run functions.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

## 2. Load the data

In [ ]:
data = pd.read_csv(r"C:\Users\adars\Downloads\datasets_newyork.csv")
data.head()

## 3. Initial inspection

Before touching anything, understand the shape, dtypes, and missingness of the raw data.

In [ ]:
print(data.shape)
data.info()

In [ ]:
data.isnull().sum()[data.isnull().sum() > 0]

## 4. Handle missing data

**Reasoning:** `neighbourhood`, `latitude`, `longitude`, etc. are all missing on the *same* 7 rows — a sign those rows are essentially blank, not worth salvaging. `price` is separately missing on 34 rows (including those same 7). Before dropping anything, check whether the missingness is random — if it disproportionately hits one group, dropping could quietly bias later comparisons.

In [ ]:
# check whether missing price skews toward a particular borough before dropping
missing_price = data[data['price'].isna()]
print('Missing price count:', len(missing_price))
print(missing_price['neighbourhood_group'].value_counts(normalize=True).round(3) * 100)
print('Overall borough split for comparison:')
print(data['neighbourhood_group'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Brooklyn is over-represented among missing-price rows, but volume is small (~0.2% of data)
# -> dropping is acceptable here; would need imputation instead if this were a larger share
print(f'Rows before: {len(data)}')
data = data.dropna()
data = data.reset_index(drop=True)
print(f'Rows after: {len(data)}')

## 5. Remove true duplicates

**Reasoning:** distinguish *true* full-row duplicates (safe to drop, no judgment needed) from rows that merely share a corrupted `id` (NOT the same thing — dropping those would destroy distinct real listings). `duplicated()` with no `subset` only flags rows where *every* column matches, so it's safe here.

In [ ]:
print('True full-row duplicates:', data.duplicated().sum())
data[data.duplicated(keep=False)].sort_values(by=data.columns[0]).head(4)

In [ ]:
data = data.drop_duplicates()
data = data.reset_index(drop=True)
print('Duplicates remaining:', data.duplicated().sum())
print('Row count:', len(data))

## 6. Fix `id` and `host_id` dtype

**Reasoning:** these are identifiers, not quantities — leaving them numeric risks accidental math (`.mean()` on an ID is meaningless) and further float precision loss. `id` is already corrupted (Excel scientific-notation damage) so it's routed through `Int64` first to strip the trailing `.0` before converting to text — this doesn't fix the corruption, but stops it from getting worse.

In [ ]:
data['id'] = data['id'].astype('Int64').astype(str)
data['host_id'] = data['host_id'].astype(str)
data[['id', 'host_id']].dtypes

## 7. Clean `rating`, `bedrooms`, `baths`

**Reasoning:** each column is text only because of specific placeholder strings, and each placeholder means something different:
- `rating`: `'No rating'` and `'New'` both mean *no score exists yet* → should become `NaN`, not `0` (0 would falsely imply the worst possible rating).
- `bedrooms`: `'Studio'` is a genuine value meaning zero separate bedrooms → becomes `0`, not `NaN`.
- `baths`: `'Not specified'` is genuinely missing → `NaN`. A literal `'0'` baths is kept as-is (unusual but plausible for some listing types).

In [ ]:
data['rating'] = data['rating'].replace(['No rating', 'New'], pd.NA).astype(float)
data['bedrooms'] = data['bedrooms'].replace('Studio', 0).astype(float)
data['baths'] = data['baths'].replace('Not specified', pd.NA).astype(float)

print(data[['rating', 'bedrooms', 'baths']].dtypes)
data[['rating', 'bedrooms', 'baths']].describe()

## 8. Feature engineering

**Reasoning:**
- `price_per_bed` normalizes price by listing size — checked `beds` has no zeros first, so no division-by-zero risk.
- `price_flag` / `min_nights_flag` mark extreme, likely-erroneous values (suspicious round-number prices like $10,000+, minimum stays over a year) WITHOUT deleting the rows — this preserves every other column's data while letting price-sensitive analysis exclude them when needed.
- `is_30_night_min` and `license_status` set up the Local Law 18 hypothesis test in the next section.

In [ ]:
# sanity check before dividing
print('Zero-bed listings:', (data['beds'] == 0).sum())

data['price_per_bed'] = data['price'] / data['beds']
data['price_flag'] = data['price'] >= 10000
data['min_nights_flag'] = data['minimum_nights'] > 365
data['is_30_night_min'] = data['minimum_nights'] == 30

data['license_status'] = data['license'].apply(
 lambda x: 'No License' if x == 'No License' else ('Exempt' if x == 'Exempt' else 'Licensed')
)

print('Flagged extreme prices:', data['price_flag'].sum())
print('Flagged extreme min_nights:', data['min_nights_flag'].sum())
print(data['license_status'].value_counts())

## 9. Cleaning summary

In [ ]:
print(f'''
Cleaning summary
----------------
Started with 20,770 rows
Dropped 34 rows (missing price, including 7 nearly-empty rows)
Removed 12 duplicate pairs (24 rows)
Final row count: {len(data)}
id/host_id converted to text (protects against corrupted-ID math errors)
rating/bedrooms/baths converted from text to numeric (Studio -> 0, placeholders -> NaN)
Added price_flag and min_nights_flag for extreme values (kept, not deleted)
Added price_per_bed, is_30_night_min, license_status as engineered features
''')

## 10. Univariate analysis — `price`

A viz-only filtered copy (`data_viz`) is used so extreme outliers don't crush the plot — the real `data` table stays untouched.

In [ ]:
data_viz = data[data['price'] < 1500]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=data_viz, x='price', ax=axes[0])
axes[0].set_title('Price — Boxplot (capped at $1500)')

sns.histplot(data=data_viz, x='price', bins=100, ax=axes[1])
axes[1].set_title('Price — Distribution')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 11. Univariate analysis — `availability_365`

Boxplot alone looks unremarkable (wide box, no outliers) — the histogram reveals the real story: bimodal clustering at 0 (fully blocked) and 365 (fully open).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=data, x='availability_365', bins=30, ax=axes[0])
axes[0].set_title('Availability 365 — Distribution')

sns.boxplot(data=data, x='availability_365', ax=axes[1])
axes[1].set_title('Availability 365 — Boxplot')
plt.tight_layout()
plt.show()

## 12. Bivariate analysis

In [ ]:
print('Median price by borough:')
print(data.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=data_viz, x='neighbourhood_group', y='price', hue='room_type')
plt.title('Median Price by Borough, split by Room Type')
plt.ylabel('Price ($)')
plt.xlabel('Borough')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=data_viz, x='number_of_reviews', y='price', hue='neighbourhood_group', alpha=0.4)
plt.title('Price vs Number of Reviews, by Borough')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(
 data=data_viz,
 vars=['price', 'minimum_nights', 'number_of_reviews', 'availability_365'],
 hue='room_type',
 plot_kws={'alpha': 0.3, 's': 15}
)
plt.show()

In [ ]:
plt.figure(figsize=(9, 8))
sns.scatterplot(data=data, x='longitude', y='latitude', hue='neighbourhood_group', s=8, alpha=0.4)
plt.title('Listing Locations by Borough')
plt.tight_layout()
plt.show()

## 13. Hypothesis test — does license status relate to the 30-night minimum?

**Reasoning:** 81% of listings default to a 30-night minimum stay, which looks like a fingerprint of NYC's Local Law 18 (hosts avoiding short-term rental restrictions). If that's true, unlicensed/exempt listings should cluster more heavily at 30 nights than licensed ones.

In [ ]:
pd.crosstab(data['license_status'], data['is_30_night_min'], normalize='index').round(3) * 100

## 14. Correlation heatmap

**Caveat worth stating explicitly:** correlation only captures *linear* relationships. The 30-night clustering and 0/365 availability bimodality are real, meaningful patterns that won't show up strongly here, since they're threshold/categorical effects, not straight-line trends.

In [ ]:
cols = ['price', 'price_per_bed', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'beds', 'rating']

corr_matrix = data[cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

## 15. Load to PostgreSQL

**Security note:** the password is read from an environment variable rather than hardcoded, so it never sits in plain text in this notebook — important once this goes on GitHub for your portfolio.

Set it once in your terminal before launching Jupyter:
```
setx POSTGRES_PASSWORD "your_actual_password"
```
(Windows — close and reopen your terminal/Jupyter after running this once.)

In [ ]:
from sqlalchemy import create_engine, text
import os

username = 'postgres'
password = os.environ.get('POSTGRES_PASSWORD')
host = 'localhost'
port = '5432'
database = 'airbnb_ny_listing'

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

# quick connection check
with engine.connect() as conn:
 result = conn.execute(text('SELECT version();'))
 print(result.fetchone())

In [ ]:
table_name = 'airbnbny'

data.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")
print(f"Rows loaded: {len(data)}")

In [ ]:
# verify the load
check = pd.read_sql(f'SELECT * FROM {table_name} LIMIT 5', engine)
check